In [60]:
import torch
import torch.nn as nn

In [61]:
model = nn.Sequential(
    nn.Linear(1, 1),
    nn.Sigmoid(),
    nn.Linear(1, 1),
    nn.Identity()
)

for i in [0, 2]:
  _ = nn.init.constant_(model[i].weight, 1)
  _ = nn.init.constant_(model[i].bias, 0)

for name, param in model.named_parameters():
  print(name, param, '\n')

0.weight Parameter containing:
tensor([[1.]], requires_grad=True) 

0.bias Parameter containing:
tensor([0.], requires_grad=True) 

2.weight Parameter containing:
tensor([[1.]], requires_grad=True) 

2.bias Parameter containing:
tensor([0.], requires_grad=True) 



In [62]:
model(torch.tensor([0.]))
model(torch.tensor([1.]))

tensor([0.5000], grad_fn=<ViewBackward0>)

tensor([0.7311], grad_fn=<ViewBackward0>)

## Creating a custom model

Alternative of  `nn.Sequential()`

In [63]:
class MyCustomModel(nn.Module):
  def __init__(self):
    super().__init__()

    self.layer1 = nn.Linear(1, 1)
    self.sigmoid = nn.Sigmoid()
    self.layer2 = nn.Linear(1, 1)
    self.identity = nn.Identity()

    for m in [self.layer1, self.layer2]:
      nn.init.constant_(m.weight, 1.0)
      nn.init.constant_(m.bias, 0.0)

  def forward(self, x):
    # Linear -> Sigmoid -> Linear -> Identity
    x = self.layer1(x)
    x = self.sigmoid(x)
    x = self.layer2(x)
    x = self.identity(x)
    return x


model = MyCustomModel()
print("Output for 0.0:", model(torch.tensor([0.])))
print("Output for 1.0:", model(torch.tensor([1.])))

Output for 0.0: tensor([0.5000], grad_fn=<ViewBackward0>)
Output for 1.0: tensor([0.7311], grad_fn=<ViewBackward0>)


## Creating a custom layer

Alternative of  `nn.Linear()`

In [64]:
class MySimpleLinear(nn.Module):
  def __init__(self, in_features, out_features):
    super().__init__()
    self.weight = nn.Parameter(torch.ones(in_features, out_features))
    self.bias = nn.Parameter(torch.zeros(out_features))

  def forward(self, x):
    return x @ self.weight + self.bias


model = nn.Sequential(
    MySimpleLinear(1, 1),
    nn.Sigmoid(),
    MySimpleLinear(1, 1),
    nn.Identity()
)

print(f"Output for 0.0: {model(torch.tensor([0.]))}")
print(f"Output for 1.0: {model(torch.tensor([1.]))}")

Output for 0.0: tensor([0.5000], grad_fn=<AddBackward0>)
Output for 1.0: tensor([0.7311], grad_fn=<AddBackward0>)


## Looking into Autograd

In [91]:
x = torch.tensor([2.], requires_grad=True)
y = x ** 3
y.backward()
x.grad

# Manually calculate derivative (Using chain rule)
3 * (x) ** 2

tensor([12.])

tensor([12.], grad_fn=<MulBackward0>)

In [93]:
x = torch.tensor([2.], requires_grad=True)
y = (5*x) ** 3
y.backward()
x.grad

# Manually calculate derivative (Using chain rule)
(3 * (5*x) ** 2) * (5)

tensor([1500.])

tensor([1500.], grad_fn=<MulBackward0>)

Gradient of many values at once

In [77]:
x = torch.tensor([2., 5., 20], requires_grad=True)
y = x ** 3
y.backward(gradient=torch.tensor([1, 1, 1]))
x.grad

tensor([  12.,   75., 1200.])

In [78]:
x = torch.tensor([2., 5., 20], requires_grad=True)
y = x ** 3
y.backward(gradient=torch.tensor([2, 2, 2]))
x.grad

tensor([  24.,  150., 2400.])

In [79]:
x = torch.tensor([2., 5., 20], requires_grad=True)
y = x ** 3
y.backward(gradient=torch.tensor([0, 0, 0]))
x.grad

tensor([0., 0., 0.])

## Squeeze and Stack

In [104]:
a = torch.tensor([[1], [2], [3]])
b = torch.tensor([4, 5, 6])
a.squeeze()
b.unsqueeze(1)
try:
  torch.stack([a.squeeze(), b.unsqueeze(1)], dim=1)
except Exception as e:
  print(f'💀 {e}')

tensor([1, 2, 3])

tensor([[4],
        [5],
        [6]])

💀 stack expects each tensor to be equal size, but got [3] at entry 0 and [3, 1] at entry 1
